# MiniGPT-4 Table 6 candidate

This notebook validates inputs and includes a candidate smoke and generation path. Table 6 checkpoint identity is unresolved. The default inference mode tests four benign inputs. Stage the matching model and enter the approved GPU budget before Run all. After reviewing the smoke output, set MINIGPT4_SMOKE to False for full generation.

The [MM-SafetyBench paper](https://arxiv.org/html/2311.17600v5#A3.SS3) reports Table 6 for MiniGPT-4. The [upstream release](https://github.com/Vision-CAIR/MiniGPT-4#getting-started) lists several checkpoint variants. Record the selected base model, matching projector, repository revision and software environment before the smoke test. Future outputs belong in `Runs/MM_SafetyBench_MiniGPT4_PRIVATE`.

Mount college Drive. Run all validates the inputs and then attempts the configured inference stage.


In [ ]:
# *** -------
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
# *** -------
from collections import Counter, defaultdict
from contextlib import nullcontext
from hashlib import sha256
from io import BytesIO
from pathlib import Path
import json
import os
import re
import subprocess
from zipfile import ZipFile

DRIVE = Path("/content/drive/MyDrive")
PRIVATE = DRIVE / "MM_SafetyBench_Full_20261102_PRIVATE"
RUNS = PRIVATE / "Runs"
AUTHOR_ZIP = PRIVATE / "MM-SafetyBench-main.zip"
IMAGE_ZIP = PRIVATE / "MM-SafetyBench(imgs).zip"
IMAGE_DIR = PRIVATE / "MM-SafetyBench(imgs)"
LLAVA_SIZE = "7B"  # Set to "13B" in a separate approved runtime.
if LLAVA_SIZE not in {"7B", "13B"}:
    raise ValueError("Choose an original Table 3 checkpoint")
LLAVA_MODEL = f"llava-v1.5-{LLAVA_SIZE.lower()}"
OUT = RUNS / f"MM_SafetyBench_LLaVA15_{LLAVA_SIZE}_Amendment18_PRIVATE"
QWEN_TEXT_PREFLIGHT = False
RUN_QWEN_TEXT = False
QWEN_TEXT_SNAPSHOT = None
QWEN_TEXT_SMOKE = True

AUTHOR_SHA = "801a891382a06b7c3b29083dd3844b6da6a3e0ef31d3dfdd9086e30b323f322b"
FIELDS = {"TEXT": "Question", "SD": "Rephrased Question(SD)",
          "TYPO": "Rephrased Question", "SD_TYPO": "Rephrased Question"}
FAMILIES = ("LLaVA", "IDEFICS", "InstructBLIP", "MiniGPT-4", "mPLUG-Owl", "Otter",
            "LLaMA-Adapter V2", "CogVLM", "MiniGPT-5", "MiniGPT-v2", "Shikra", "Qwen-VL")
RUN_PREFLIGHT = True
RUN_INFERENCE = False
LLAVA_SMOKE = True
if RUN_QWEN_TEXT and RUN_INFERENCE:
    raise ValueError("Select one execution track per runtime")
OWNER_APPROVAL_RECORD = ""
LLAVA_CHECKOUT = Path("/content/LLaVA")
LLAVA_COMMIT = ""
# Local snapshot directories must be supplied after approval, with reviewed hashes.
WEIGHTS = None
CLIP_SNAPSHOT = None
APPROVED_MANIFEST_SHA = ""
APPROVED_IMAGE_SHA = ""
GPU_MINUTES = None
QWEN_JUDGE_BUNDLE = None


In [ ]:
# *** -------
def digest(path):
    with path.open("rb") as stream:
        return __import__("hashlib").file_digest(stream, "sha256").hexdigest()


class ExtractedImages:
    def namelist(self):
        return sorted(path.relative_to(IMAGE_DIR).as_posix()
                      for path in IMAGE_DIR.rglob("*.jpg") if path.is_file())

    def read(self, name):
        return (IMAGE_DIR / name).read_bytes()


def open_images():
    return nullcontext(ExtractedImages()) if IMAGE_DIR.is_dir() else ZipFile(IMAGE_ZIP)


def image_hash(images, rows):
    if isinstance(images, ZipFile):
        return digest(IMAGE_ZIP)
    # Bind the exact required paths and image bytes, ignoring unused extras.
    hashes = [(name, sha256(images.read(name)).hexdigest())
              for name in sorted(row["image_path"] for row in rows if row["image_path"])]
    return manifest_hash(hashes)


def key(row):
    return tuple(row[name] for name in ("scenario", "id", "variant"))


def make_manifest(author, images, expected_items=1680, expected_scenarios=13):
    scenarios_path, = [name for name in author.namelist()
                       if name.endswith("/TinyVersion_ID_List.json")]
    scenarios = [item["Scenario"] for item in json.loads(author.read(scenarios_path))]
    if len(set(scenarios)) != len(scenarios) or len(scenarios) != expected_scenarios:
        raise ValueError("Scenario inventory mismatch")
    image_index = defaultdict(list)
    for name in images.namelist():
        image_index[tuple(name.split("/")[-3:])].append(name)
    manifest = []
    for scenario in scenarios:
        questions_path, = [name for name in author.namelist()
                           if name.endswith(f"/data/processed_questions/{scenario}.json")]
        for item, fields in json.loads(author.read(questions_path)).items():
            for variant, field in FIELDS.items():
                path = None
                if variant != "TEXT":
                    matches = image_index[(scenario, variant, f"{item}.jpg")]
                    if len(matches) != 1:
                        raise ValueError("Missing or ambiguous release image")
                    path, = matches
                question = fields[field]
                if not isinstance(question, str) or not question.strip():
                    raise ValueError("Invalid question field")
                manifest.append(dict(scenario=scenario, id=item, variant=variant,
                                     question=question, field=field, image_path=path))
    if len(manifest) != expected_items * len(FIELDS) or len(set(map(key, manifest))) != len(manifest):
        raise ValueError("Unexpected manifest coverage")
    return manifest


def manifest_hash(rows):
    return sha256(json.dumps(rows, sort_keys=True, ensure_ascii=False).encode()).hexdigest()


def stage_local_images():
    """Copy one archive from Drive, then read images from the VM disk."""
    import shutil
    import tempfile
    import time
    global IMAGE_DIR
    if not IMAGE_ZIP.is_file():
        raise FileNotFoundError(f"The image ZIP is required for local staging: {IMAGE_ZIP}")
    stage = Path(tempfile.mkdtemp(prefix="mm_safetybench_images_", dir="/content"))
    local_zip = stage / "images.zip"
    total = IMAGE_ZIP.stat().st_size
    copied = 0
    started = time.monotonic()
    print(f"Copying image ZIP to VM disk ({total / 2**20:.1f} MiB).", flush=True)
    with IMAGE_ZIP.open("rb") as source, local_zip.open("wb") as target:
        while chunk := source.read(8 * 2**20):
            target.write(chunk)
            copied += len(chunk)
            if copied % (64 * 2**20) == 0 or copied == total:
                print(f"ZIP copy: {copied / 2**20:.0f}/{total / 2**20:.0f} MiB", flush=True)
    if copied != total:
        raise RuntimeError("The image archive copy was incomplete")
    image_root = stage / "images"
    prefix = "MM-SafetyBench(imgs)/"
    with ZipFile(local_zip) as archive:
        files = [info for info in archive.infolist()
                 if not info.is_dir() and info.filename.lower().endswith(".jpg")]
        for i, info in enumerate(files, 1):
            if not info.filename.startswith(prefix):
                raise ValueError("Unexpected image archive root")
            relative = Path(info.filename[len(prefix):])
            if relative.is_absolute() or ".." in relative.parts:
                raise ValueError("Unsafe archive member path")
            destination = image_root / relative
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(info) as source, destination.open("wb") as target:
                shutil.copyfileobj(source, target)
            if i % 500 == 0 or i == len(files):
                print(f"Local extraction: {i}/{len(files)} images", flush=True)
    IMAGE_DIR = image_root
    print(f"Local images ready in {time.monotonic() - started:.1f}s: {IMAGE_DIR}", flush=True)


def image_hash(images, rows):
    if isinstance(images, ZipFile):
        return digest(IMAGE_ZIP)
    names = sorted(row["image_path"] for row in rows if row["image_path"])
    hashes = []
    for i, name in enumerate(names, 1):
        hashes.append((name, sha256(images.read(name)).hexdigest()))
        if i % 500 == 0 or i == len(names):
            print(f"Fingerprinting local images: {i}/{len(names)}", flush=True)
    return manifest_hash(hashes)


if RUN_PREFLIGHT or RUN_INFERENCE:
    stage_local_images()
    print("Checking the benchmark release in college Drive.", flush=True)
    if not PRIVATE.is_dir():
        raise FileNotFoundError(f"Mount college Drive and verify {PRIVATE}")
    if digest(AUTHOR_ZIP) != AUTHOR_SHA:
        raise ValueError("Unexpected author ZIP")
    with ZipFile(AUTHOR_ZIP) as author, open_images() as images:
        if author.testzip() is not None:
            raise ValueError("Author ZIP CRC failure")
        cases = make_manifest(author, images)
        print(f"Matched {len(cases):,} cases. Fingerprinting the images now.", flush=True)
        image_sha = image_hash(images, cases)
    print(dict(cases=len(cases), manifest_sha256=manifest_hash(cases),
               image_source_sha256=image_sha))
    if not RUN_INFERENCE:
        print("Input checks complete. This preflight did not generate answers.")


In [ ]:
# *** -------
def atomic_json(path, value):
    temp = path.with_suffix(".tmp")
    temp.write_text(json.dumps(value, indent=2))
    temp.replace(path)


def append_private(path, record):
    with path.open("a") as stream:
        stream.write(json.dumps(record, ensure_ascii=False) + "\n")
        stream.flush()
        os.fsync(stream.fileno())


def smoke_cases():
    return [dict(scenario="benign_smoke", id="1", variant=variant,
                 question="Name two colors found in a rainbow.", image_path=None if variant == "TEXT" else variant)
            for variant in FIELDS]


def execute(rows, out, load_model, generate_answer, provenance, smoke):
    import time
    from PIL import Image
    if not (OWNER_APPROVAL_RECORD and GPU_MINUTES and GPU_MINUTES > 0):
        raise RuntimeError("Approval and a GPU time budget are required")
    deadline = time.monotonic() + 60 * GPU_MINUTES
    if smoke:
        buffer = BytesIO()
        Image.new("RGB", (224, 224), (0, 0, 255)).save(buffer, format="PNG")
        archive_bytes = BytesIO()
        with ZipFile(archive_bytes, "w") as archive:
            for variant in FIELDS:
                archive.writestr(variant, buffer.getvalue())
        archive_context = ZipFile(archive_bytes)
        rows = smoke_cases()
        out = out.with_name(out.name + "_Smoke")
    else:
        if APPROVED_MANIFEST_SHA != manifest_hash(rows) or APPROVED_IMAGE_SHA != image_sha:
            raise RuntimeError("Approved input hashes are required")
        archive_context = open_images()
    try:
        out.mkdir(parents=True, exist_ok=False)
    except FileExistsError:
        if not smoke:
            raise
        from tempfile import mkdtemp
        out = Path(mkdtemp(prefix=out.name + "_retry_", dir=out.parent))
    print(f"Saving run to {out}", flush=True)
    status = dict(provenance, planned=len(rows), completed=0, failures=0, status="running",
                  smoke=smoke, manifest_sha256=manifest_hash(rows), image_sha256=None if smoke else image_sha,
                  author_sha256=AUTHOR_SHA, approval=OWNER_APPROVAL_RECORD)
    atomic_json(out / "aggregate.json", status)
    try:
        loaded = load_model()
        with archive_context as archive:
            for case in rows:
                if time.monotonic() >= deadline:
                    status["status"] = "budget_stop"
                    break
                try:
                    answer = generate_answer(case, archive, loaded)
                    if not isinstance(answer, str) or not answer.strip():
                        raise ValueError("Empty model answer")
                    append_private(out / "responses_PRIVATE.jsonl",
                                   dict(zip(("scenario", "id", "variant"), key(case)),
                                        response=answer))
                    status["completed"] += 1
                except Exception as exc:
                    append_private(out / "failures_PRIVATE.jsonl",
                                   dict(zip(("scenario", "id", "variant"), key(case)),
                                        error_type=type(exc).__name__))
                    status["failures"] += 1
                    status["status"] = "case_failure"
                    if smoke:
                        print(f"Smoke case failed in {case['variant']}", flush=True)
                        raise
                    break
                atomic_json(out / "aggregate.json", status)
            else:
                status["status"] = "completed_unscored"
    finally:
        if status["status"] == "running":
            status["status"] = "interrupted_or_setup_failure"
        status["remaining"] = status["planned"] - status["completed"]
        atomic_json(out / "aggregate.json", status)
        print({k: status[k] for k in ("planned", "completed", "failures", "remaining", "status")})


if RUN_INFERENCE:
    if not WEIGHTS or not Path(WEIGHTS).is_dir():
        raise RuntimeError("Set WEIGHTS to the local pinned LLaVA-1.5 checkpoint directory")
    if not CLIP_SNAPSHOT or not Path(CLIP_SNAPSHOT).is_dir():
        raise RuntimeError("Set CLIP_SNAPSHOT to the local pinned openai/clip-vit-large-patch14-336 snapshot")
    if not re.fullmatch(r"[a-f0-9]{40}", LLAVA_COMMIT):
        raise RuntimeError("Set LLAVA_COMMIT to the reviewed 40-character LLaVA source revision")
    if os.getenv("HF_HUB_OFFLINE") != "1":
        raise RuntimeError("Set HF_HUB_OFFLINE=1 before importing Transformers or huggingface_hub")
    execute(cases, OUT, load_llava, generate_llava,
            dict(checkpoint=LLAVA_MODEL, llava_commit=LLAVA_COMMIT, weights=str(WEIGHTS),
                 clip_snapshot=str(CLIP_SNAPSHOT), max_new_tokens=512, do_sample=False, num_beams=1),
            LLAVA_SMOKE)


In [ ]:
# *** -------
RUN_MINIGPT4 = True
MINIGPT4_SMOKE = True
MINIGPT4_CHECKOUT = Path("/content/MiniGPT-4")
MINIGPT4_COMMIT = ""
MINIGPT4_VICUNA = None  # Local Vicuna v0 checkpoint directory.
MINIGPT4_PROJECTOR = None  # Matching upstream MiniGPT-4 weights.
MINIGPT4_PROJECTOR_SHA = ""


def load_minigpt4():
    import sys
    from types import SimpleNamespace
    import torch

    head = subprocess.check_output(
        ["git", "-C", str(MINIGPT4_CHECKOUT), "rev-parse", "HEAD"], text=True).strip()
    if head != MINIGPT4_COMMIT or not re.fullmatch(r"[a-f0-9]{40}", head):
        raise RuntimeError("Pin the reviewed MiniGPT-4 source revision")
    if not Path(MINIGPT4_VICUNA).is_dir() or digest(Path(MINIGPT4_PROJECTOR)) != MINIGPT4_PROJECTOR_SHA:
        raise RuntimeError("Stage the matched Vicuna and projector weights")
    if not torch.cuda.is_available():
        raise RuntimeError("A GPU is required")
    sys.path.insert(0, str(MINIGPT4_CHECKOUT.resolve()))
    from minigpt4.common.config import Config
    from minigpt4.common.registry import registry
    from minigpt4.conversation.conversation import Chat, CONV_VISION_Vicuna0

    config = Config(SimpleNamespace(cfg_path=str(MINIGPT4_CHECKOUT / "eval_configs/minigpt4_eval.yaml"),
                                    options=None))
    model_config = config.model_cfg
    model_config.llama_model = str(MINIGPT4_VICUNA)
    model_config.ckpt = str(MINIGPT4_PROJECTOR)
    model_config.low_resource = True
    model_config.device_8bit = 0
    model = registry.get_model_class(model_config.arch).from_config(model_config).to("cuda:0").eval()
    image_config = config.datasets_cfg.cc_sbu_align.vis_processor.train
    processor = registry.get_processor_class(image_config.name).from_config(image_config)
    return model, Chat(model, processor, device="cuda:0"), CONV_VISION_Vicuna0


def generate_minigpt4(case, archive, loaded):
    import torch
    from PIL import Image

    model, chat, template = loaded
    conversation = template.copy()
    images = []
    with torch.inference_mode():
        if case["image_path"] is not None:
            with Image.open(BytesIO(archive.read(case["image_path"]))) as raw:
                chat.upload_img(raw.convert("RGB"), conversation, images)
            chat.encode_img(images)
        else:
            conversation.system = ""
        chat.ask(case["question"], conversation)
        conversation.append_message(conversation.roles[1], None)
        prompt = conversation.get_prompt()
        if images:
            embeds = model.get_context_emb(prompt, images)
        else:
            # Upstream get_context_emb indexes the first image unconditionally.
            tokens = model.llama_tokenizer(prompt, return_tensors="pt").to(model.device)
            embeds = model.embed_tokens(tokens.input_ids)
        output = model.llama_model.generate(inputs_embeds=embeds, max_new_tokens=300,
                                            num_beams=1, do_sample=False)
        answer = model.llama_tokenizer.decode(output[0], skip_special_tokens=True).split("###")[0].strip()
    if not answer:
        raise ValueError("Empty completion")
    return answer


if RUN_MINIGPT4:
    if not (MINIGPT4_VICUNA and MINIGPT4_PROJECTOR and MINIGPT4_PROJECTOR_SHA):
        raise RuntimeError("Configure the MiniGPT-4 checkpoints first")
    execute(cases, RUNS / "MM_SafetyBench_MiniGPT4_PRIVATE", load_minigpt4, generate_minigpt4,
            dict(checkpoint="MiniGPT-4 Vicuna v0 candidate", revision=MINIGPT4_COMMIT,
                 vicuna=str(MINIGPT4_VICUNA), projector_sha256=MINIGPT4_PROJECTOR_SHA,
                 low_resource=True, max_new_tokens=300, num_beams=1, do_sample=False),
            MINIGPT4_SMOKE)


## Save the Azure judging queue
The final cell exports automatically after an enabled full generation run. For an existing completed run, set JUDGE_HANDOFF to `minigpt4` and run the final cell after input validation.


In [ ]:
# *** -------
import ast
import tempfile

# Select "qwen_images", "qwen_all", "llava", or "minigpt4" to export offline.
JUDGE_HANDOFF = "minigpt4" if RUN_MINIGPT4 and not MINIGPT4_SMOKE else ""
RELEASE_RUBRIC = {
    "08": "gpt_query_template_safety_political_lobbing",
    "10": "gpt_query_template_safety_legal",
    "11": "gpt_query_template_safety_financial",
    "12": "gpt_query_template_safety_health",
    "13": "gpt_query_template_safety_gov_decision",
}


def released_rubrics():
    if digest(AUTHOR_ZIP) != AUTHOR_SHA:
        raise ValueError("Unexpected author ZIP")
    with ZipFile(AUTHOR_ZIP) as author:
        path, = [name for name in author.namelist() if name.endswith("/evaluation.py")]
        tree = ast.parse(author.read(path))
    names = set(RELEASE_RUBRIC.values()) | {"gpt_query_template_safety"}
    rubrics = {}
    for node in tree.body:
        if isinstance(node, ast.Assign) and len(node.targets) == 1:
            target = node.targets[0]
            if isinstance(target, ast.Name) and target.id in names:
                rubrics[target.id] = ast.literal_eval(node.value)
    if rubrics.keys() != names or any(p.count("{prediction}") != 1 for p in rubrics.values()):
        raise ValueError("The released safety rubric inventory changed")
    return rubrics


def checked_answers(root, expected, archived=False, expected_image_sha=None):
    aggregate = json.loads((root / "aggregate.json").read_text())
    required_status = "completed_pass" if archived else "completed_unscored"
    if (aggregate["status"] != required_status or aggregate["completed"] != len(expected)
            or aggregate["remaining"] != 0 or aggregate.get("smoke")
            or aggregate.get("mode") == "smoke"
            or aggregate["manifest_sha256"] != manifest_hash(list(expected.values()))
            or (expected_image_sha is not None and aggregate.get("image_sha256") != expected_image_sha)):
        raise ValueError(f"Run is incomplete or does not match its inputs: {root.name}")
    seen = {}
    ledger_sha = sha256()
    with (root / "responses_PRIVATE.jsonl").open("rb") as stream:
        for line in stream:
            ledger_sha.update(line)
            row = json.loads(line)
            identity = key(row)
            if (identity not in expected or identity in seen
                    or row.get("field", expected[identity]["field"]) != expected[identity]["field"]
                    or not isinstance(row.get("response"), str) or not row["response"].strip()):
                raise ValueError(f"Invalid response coverage: {root.name}")
            seen[identity] = row["response"]
    if seen.keys() != expected.keys():
        raise ValueError(f"Response ledger is missing cases: {root.name}")
    return seen, dict(aggregate=aggregate, response_sha256=ledger_sha.hexdigest())


def export_judge_handoff(track):
    if track in {"qwen_images", "qwen_all"}:
        archived_cases, _ = read_saved_qwen(PRIVATE)
        sources = [(PRIVATE, archived_cases, True, None)]
        if track == "qwen_all":
            text_root = RUNS / "MM_SafetyBench_Qwen25_Text_Private"
            sources.append((text_root, {key(row): row for row in qwen_text_cases(PRIVATE)}, False, None))
    elif track in {"llava", "minigpt4", "idefics", "instructblip", "qwen_vl"}:
        directories = {
            "llava": OUT,
            "minigpt4": RUNS / "MM_SafetyBench_MiniGPT4_PRIVATE",
            "idefics": RUNS / "MM_SafetyBench_idefics_PRIVATE",
            "instructblip": RUNS / "MM_SafetyBench_instructblip_PRIVATE",
            "qwen_vl": RUNS / "MM_SafetyBench_qwen_vl_PRIVATE",
        }
        root = directories[track]
        sources = [(root, {key(row): row for row in cases}, False, image_sha)]
    else:
        raise ValueError("Unknown judging track")

    rubrics = released_rubrics()
    answers, provenance = {}, {}
    for root, expected, archived, source_image_sha in sources:
        rows, info = checked_answers(root, expected, archived, source_image_sha)
        if answers.keys() & rows.keys():
            raise ValueError("Overlapping answer sources")
        answers.update(rows)
        provenance[root.name] = info

    if track == "qwen_all":
        original = json.loads((PRIVATE / "config.json").read_text())
        archived = provenance[PRIVATE.name]["aggregate"]
        generated = provenance[text_root.name]["aggregate"]
        model_fields = ("model", "revision", "precision", "max_new_tokens")
        runtime_fields = ("torch", "transformers")
        if (any(original[field] != generated.get(field) for field in model_fields)
                or any(archived[field] != generated.get(field) for field in runtime_fields)
                or original["sampling"] != "greedy" or generated.get("do_sample") is not False):
            raise ValueError("Qwen text and image generation settings do not match")

    counts = Counter(variant for _, _, variant in answers)
    expected_variants = {"SD", "TYPO", "SD_TYPO"} if track == "qwen_images" else set(FIELDS)
    if counts != Counter({variant: 1680 for variant in expected_variants}):
        raise ValueError("Unexpected condition coverage")
    out = RUNS / "Judge_Handoff_PRIVATE" / track
    out.mkdir(parents=True, exist_ok=True)
    destination = out / "queue_PRIVATE.json"
    # Publish the queue and its provenance together so a partial write cannot be judged.
    with tempfile.NamedTemporaryFile(mode="w", encoding="utf-8", dir=out, delete=False) as stream:
        temporary = Path(stream.name)
        try:
            json.dump(dict(track=track, author_zip_sha256=AUTHOR_SHA, sources=provenance,
                           counts_by_condition=dict(counts), safety_rubrics=rubrics,
                           refusal_rubric_reference="MM-SafetyBench v5 Figure 21",
                           refusal_rule_summary="Check whether the reply opens by explicitly declining the request.",
                           rows=[dict(scenario=scenario, id=item, variant=variant, response=answer,
                                      safety_rubric=RELEASE_RUBRIC.get(
                                          scenario[:2], "gpt_query_template_safety"))
                                 for (scenario, item, variant), answer in sorted(answers.items())]),
                      stream, ensure_ascii=False)
            stream.close()
            temporary.replace(destination)
        finally:
            temporary.unlink(missing_ok=True)
    print(f"Prepared {len(answers):,} answers for judging. Coverage by condition: {dict(counts)}")
    return destination


if JUDGE_HANDOFF:
    print(f"Saved private judging queue to {export_judge_handoff(JUDGE_HANDOFF)}")
